## 네이버 블로그 크롤링

In [55]:
# 라이브러리 불러오기
from selenium import webdriver # 브라우저 자동화를 위한 모듈
from bs4 import BeautifulSoup as BS # HTML 내용 파싱을 위한 모듈
import pandas as pd # 데이터 조작 및 분석을 모듈
import time # 코드 실행 속도 조절을 위한 모듈
from selenium.webdriver.common.by import By # 다양한 방법으로 엘리먼트를 찾기 위한 모듈
from selenium.webdriver.support.ui import WebDriverWait # 브라우저가 요소를 찾을 때까지 대기(Wait)해주는 모듈
from selenium.webdriver.support import expected_conditions as EC # '어떤 상태'가 될 때까지 기다릴지 조건을 정해주는(EC) 기능 임포트

In [56]:
# 브라우저 실행
driver = webdriver.Chrome()
# 브라우저 창 최대화 - 좋아요 수(공감 수)원활한 수집을 위해 : 최대화 안했을 때에 인식이 안됐던 점 보완
driver.maximize_window()

In [57]:
# 최근 1주일(2026.03.13~2026.03.20) 간 "미국 이란 전쟁"에 대한 네이버 블로그 검색 결과
url = 'https://search.naver.com/search.naver?ssc=tab.blog.all&query=%EB%AF%B8%EA%B5%AD%20%EC%9D%B4%EB%9E%80%20%EC%A0%84%EC%9F%81&sm=tab_opt&nso=so%3Ar%2Cp%3Afrom20260313to20260320'
driver.get(url)
time.sleep(2) # 검색 결과 로딩 대기

In [58]:
import datetime # 날짜와 시간을 다루는 파이썬 기본 라이브러리

# 네이버 검색 결과의 무한 스크롤 특성으로 인해 처음 페이지에 접속하면 네이버는 서버 부하를 줄이기 위해 약 30개 정도의 글만 먼저 보여줌. 
# 따라서 사용자가 화면을 아래로 스크롤해야 추가로 글을 더 불러올 수 있음. -> 스코롤 다운 함수를 정의해 블로그 확보 갯수 조절 가능!

# 스크롤 다운 함수 정의
def doScrollDown(whileSeconds): 
    start = datetime.datetime.now() # 함수가 실행된 '현재 시점'의 시/분/초 가져오기
    end = start + datetime.timedelta(seconds=whileSeconds) # 입력받은 초 만큼의 시간 간격 생성
    while True:
        driver.execute_script('window.scrollTo(0, document.body.scrollHeight);') # 페이지 맨 아래로 스크롤 다운
        time.sleep(1.5) # 다음 목록이 로딩될 시간(약 1.5초)을 줌
        if datetime.datetime.now() > end: # 만약 현재 시간이 아까 계산해둔 종료 시간(end)보다 커지면 반복 break
            break 

# 3초 정도 스크롤 내리기
print("검색 결과 목록을 확장 중입니다... (약 3초 소요)")
doScrollDown(3)

검색 결과 목록을 확장 중입니다... (약 3초 소요)


In [ ]:
# 네이버가 최근 블로그 검색 결과 레이아웃을 업데이트하면서 클래스명이 무작위 문자열이 섞인 형태로 바뀐것을 F12키를 활용해 인지했습니다.
# 이런 클래스명은 네이버가 코드를 업데이트할 때마다 수시로 바뀌기 때문에, '클래스 이름' 대신 해당 클래스의 '구조'나 '속성'을 이용해 크롤링하는 것이 좋다고 느꼈습니다.

# 블로그 제목과 URL 추출

# 제목과 URL을 저장할 리스트 초기화
title_list = []
url_list = []

titles = driver.find_elements(By.CSS_SELECTOR, 'a[data-heatmap-target=".nblg"]') # <a> 태그 중에서 'data-heatmap-target' 속성값이 '.nblg'인 것만 골라내기

# 찾아낸 여러 개의 요소(titles)를 하나씩 꺼내어 반복문 돌리기
for title_element in titles:
    title_text = title_element.text
    url_link = title_element.get_attribute('href')
    if title_text and url_link: # 제목과 주소가 모두 정상적으로 존재할 때만 리스트에 추가 (빈 값 방지)
        title_list.append(title_text) # 제목 리스트에 추가
        url_list.append(url_link)     # 주소 리스트에 추가

# 중복 제거 (데이터 정제)
df_temp = pd.DataFrame({'title': title_list, 'url': url_list})
df_temp = df_temp.drop_duplicates(subset=['url'] , keep='first') # subset=['url'] = 주소(url) 컬럼을 기준으로 중복된 불로그인지 검사해"라는 뜻
# keep='first' = 만약 똑같은 주소가 여러 개 발견되면, 가장 처음에 나온 것 하나만 남기고 나머지는 버려"라는 뜻
title_list = df_temp['title'].tolist()
url_list = df_temp['url'].tolist()

print(f"✅ 총 {len(url_list)}개의 중복 없는 블로그 주소를 확보했습니다.")

✅ 총 60개의 중복 없는 블로그 주소를 확보했습니다.


In [ ]:
# 블로그 본문, 좋아요, 댓글, 이미지 및 영상 갯수 총 6개 항목 상세 수집
new_doc = []      # 1. 본문
like_cnt = []     # 2. 좋아요(공감) 수
comment_cnt = []  # 3. 댓글 수
comment_list = [] # 4. 댓글 내용
img_cnt = []      # 5. 이미지 수
div_cnt = []      # 6. 영상 수

for i in range(len(url_list)): # url_list에 저장된 주소 개수만큼 반복문을 실행
    driver.execute_script(f"window.open('{url_list[i]}')")
    driver.switch_to.window(driver.window_handles[1]) # 새로 열린 탭(인덱스 1번)으로 제어권을 전환
    time.sleep(2.5) # 페이지가 완전히 로딩될 때까지 약 2.5초 대기
    
    try:
        driver.switch_to.frame('mainFrame') # 네이버 블로그는 본문이 'mainFrame'이라는 iframe 안에 숨겨져 있어 전환이 필요
        html = driver.page_source
        soup = BS(html, 'html.parser')
        
        # 1. 본문 추출
        try:
            content = soup.select_one('.se-main-container, #postViewArea').get_text(strip=True)
        except: # 본문을 찾을 수 없는 경우
            content = "null"
        new_doc.append(content)

        # 2. 좋아요(공감) 추출
        try:
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);") # 화면 아래에 있어야 공감 버튼 요소가 활성화되는 경우가 많아 바닥으로 스크롤
            time.sleep(1)
            wait = WebDriverWait(driver, 5) # 요소가 나타날 때까지 최대 5초 대기
            like_el = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "span.u_likeit_text._count.num")))
            like_val = like_el.get_attribute('textContent').strip() # .text 대신 'textContent'를 사용 -> 숨겨진 숫자까지 강제로 가져오기
            if not like_val or not like_val.isdigit(): like_val = "0" # 숫자가 비어있거나 숫자가 아니면 '0'으로 처리
        except:
            like_val = "0"
        like_cnt.append(like_val)

        # 3. 댓글 수 추출
        try:
            comment_num = driver.find_element(By.ID, "commentCount").get_attribute('textContent').strip()
            if not comment_num: comment_num = "0"
        except:
            comment_num = "0"
        comment_cnt.append(comment_num)

        # 4. 이미지/영상 수 추출
        # BeautifulSoup의 select 기능을 사용하여 이미지 태그와 영상 관련 클래스의 개수 세기
        img_cnt.append(len(soup.select('img.se-image-resource')))
        div_cnt.append(len(soup.select('.pzp-ui-dimmed, .se-video-container')))

        # 5. 댓글 내용 추출
        try:
            comment_btn = driver.find_element(By.ID, "commentCount") # 댓글 버튼을 찾아 자바스크립트로 강제 클릭
            driver.execute_script("arguments[0].click();", comment_btn) 
            time.sleep(1) 
            comments = driver.find_elements(By.CLASS_NAME, "u_cbox_contents") # 댓글 내용이 담긴 요소들 모두 찾기
            comment_final = "\n".join([c.text for c in comments]) if comments else "댓글 없음" # 댓글 창은 정상적으로 열렸으나, 실제로 작성된 댓글이 0개인 경우
        except:
            comment_final = "null" # 블로그 주인이 댓글 기능을 아예 막아 놓은 경우 (commentCount 요소를 찾지 못함)
        comment_list.append(comment_final)

    except Exception as e:
        print(f"⚠️ {i+1}번 글 오류: {e}")
        # 오류 시에도 리스트 길이를 맞추기 위해 기본값 삽입
        for lst in [new_doc, like_cnt, comment_cnt, comment_list]: lst.append("error")
        for lst in [img_cnt, div_cnt]: lst.append(0)

    # 작업이 끝난 탭을 닫고 다시 메인 목록 탭(인덱스 0번)으로 돌아가기
    driver.close()
    driver.switch_to.window(driver.window_handles[0])
    
    # 잘 되고 있는지 10개마다 진행 상황 출력
    if (i + 1) % 10 == 0:
        print(f"진행 상황: {i+1}/{len(url_list)} 완료")

driver.quit() # 브라우저 종료
print("수집 프로세스 종료!")

진행 상황: 10/60 완료
진행 상황: 20/60 완료
진행 상황: 30/60 완료
진행 상황: 40/60 완료
진행 상황: 50/60 완료
진행 상황: 60/60 완료
수집 프로세스 종료!


In [61]:
# 데이터프레임 생성
raw_data = pd.DataFrame({
    'title': title_list,
    'doc': new_doc,
    'like': like_cnt,
    'comment_cnt': comment_cnt,
    'comment_list': comment_list,
    'img': img_cnt,
    'div': div_cnt,
    'ch': 'naver',
    'ch2': 'blog'
})

# 파일경로 설정
save_path = 'C:\\Users\\황태하\\'

# 1. 엑셀 저장
raw_data.to_excel(save_path + '블로그크롤링과제_황태하.xlsx', index=False)

# 2. CSV 저장 (한글 깨짐 방지 utf-8-sig)
raw_data.to_csv(save_path + 'naver_blog202403.csv', index=False, encoding="utf-8-sig")

print(f"파일 저장 완료! (위치: {save_path})")

파일 저장 완료! (위치: C:\Users\황태하\)


In [ ]:
# 1. 경로 및 파일명 설정
file_path = 'C:\\Users\\황태하\\'
file_name = '블로그크롤링과제_황태하.xlsx'
full_path = file_path + file_name

try:
    # 2. 엑셀 파일 읽어오기
    load_data = pd.read_excel(full_path)

    # 3. 데이터 확인 및 요약
    print("✅ 파일을 성공적으로 불러왔습니다!")
    print(f"파일 경로: {full_path}")
    print(f"총 수집 데이터: {len(load_data)}행")
    print("-" * 30)
    
    # 상위 3개 데이터 샘플 출력
    print("📋 데이터 샘플 (상위 3개):")
    print(load_data[['title', 'like', 'comment_cnt']].head(3)) # 블로그 제목, 좋아요(공감) 수, 댓글 수 3개 항목만 불러오기
    print("-" * 30)

except FileNotFoundError:
    print(f"[오류] 파일을 찾을 수 없습니다. 경로를 확인해주세요: {full_path}")
except Exception as e:
    print(f"[오류] 읽기 과정에서 문제가 발생했습니다: {e}")

print("검증 완료!!")

✅ 파일을 성공적으로 불러왔습니다!
파일 경로: C:\Users\황태하\블로그크롤링과제_황태하.xlsx
총 수집 데이터: 60행
------------------------------
📋 데이터 샘플 (상위 3개):
                                  title  like  comment_cnt
0  미국-이란 전쟁 '유가 100달러' 시대… 에너지 취약계층 그림자    37           17
1           미국·이란 전쟁 수혜주, 왜 조선주가 주목받을까?     8            7
2     미국 이란 전쟁 이유, 우리나라 경제 주가에 미치는 영향은?    41            4
------------------------------
검증 완료!!
